In [15]:
#!/usr/bin/env python3
"""
Interactive Beehive Inspection Timeline Generator
Creates Plotly visualizations for the beehive data story blog post
"""

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
from pathlib import Path

In [18]:
def generate_sample_inspection_data():
    """
    Generate realistic sample data for beehive inspections
    Based on typical beekeeping patterns over 4 years
    """
    
    # Base dates for inspections (seasonal patterns)
    base_dates = []
    
    # Generate 4 years of data (2020-2024)
    for year in range(2020, 2024):
        # Spring inspections (March-May) - checking winter survival, buildup
        spring_dates = [
            datetime(year, 3, 15),
            datetime(year, 4, 10),
            datetime(year, 4, 25),
            datetime(year, 5, 8),
            datetime(year, 5, 22)
        ]
        
        # Summer inspections (June-August) - regular monitoring
        summer_dates = [
            datetime(year, 6, 5),
            datetime(year, 6, 28),
            datetime(year, 7, 15),
            datetime(year, 8, 3),
            datetime(year, 8, 25)
        ]
        
        # Fall inspections (September-November) - harvest prep, winter prep
        fall_dates = [
            datetime(year, 9, 10),
            datetime(year, 9, 30),
            datetime(year, 10, 15),
            datetime(year, 11, 5)
        ]
        
        # Winter check (December-February) - minimal disturbance
        if year < 2023:  # Don't add future winter dates
            winter_dates = [
                datetime(year, 12, 20),
                datetime(year + 1, 1, 15),
                datetime(year + 1, 2, 28)
            ]
            base_dates.extend(winter_dates)
        
        base_dates.extend(spring_dates + summer_dates + fall_dates)
    
    # Add some randomness to dates (±3 days)
    inspection_dates = []
    for date in base_dates:
        if date <= datetime.now():  # Only past dates
            random_offset = timedelta(days=random.randint(-3, 3))
            inspection_dates.append(date + random_offset)
    
    # Sort dates
    inspection_dates.sort()
    
    # Generate data for each inspection
    inspections = []
    
    for i, date in enumerate(inspection_dates):
        # Seasonal patterns for photo counts and findings
        month = date.month
        season = get_season(month)
        
        # Photo count varies by season and events
        if season == "Spring":
            base_photos = random.randint(8, 15)  # Moderate documentation
            if random.random() < 0.3:  # 30% chance of interesting spring event
                base_photos += random.randint(5, 10)  # Queen sighting, swarm prep
        elif season == "Summer": 
            base_photos = random.randint(5, 12)  # Regular checks
            if random.random() < 0.4:  # 40% chance of honey-related photos
                base_photos += random.randint(3, 8)
        elif season == "Fall":
            base_photos = random.randint(10, 18)  # Harvest season, more documentation
            if random.random() < 0.5:  # 50% chance of harvest/prep photos
                base_photos += random.randint(5, 12)
        else:  # Winter
            base_photos = random.randint(3, 8)  # Minimal disturbance
        
        # Bee confidence varies by season and activity
        if season == "Spring":
            bee_confidence = random.uniform(0.75, 0.95)  # Active season
        elif season == "Summer":
            bee_confidence = random.uniform(0.80, 0.95)  # Peak activity
        elif season == "Fall":
            bee_confidence = random.uniform(0.70, 0.90)  # Still active but preparing
        else:  # Winter
            bee_confidence = random.uniform(0.40, 0.75)  # Clustered, less visible
        
        # Honey area estimates (seasonal)
        if season == "Spring":
            honey_area = random.uniform(0.15, 0.40)  # Building up stores
        elif season == "Summer":
            honey_area = random.uniform(0.30, 0.65)  # Peak honey season
        elif season == "Fall":
            honey_area = random.uniform(0.20, 0.45)  # Post-harvest
        else:  # Winter
            honey_area = random.uniform(0.05, 0.25)  # Consuming stores
        
        # Brood area estimates (seasonal)
        if season == "Spring":
            brood_area = random.uniform(0.25, 0.55)  # Building up population
        elif season == "Summer":
            brood_area = random.uniform(0.30, 0.60)  # Peak brood rearing
        elif season == "Fall":
            brood_area = random.uniform(0.10, 0.35)  # Slowing down
        else:  # Winter
            brood_area = random.uniform(0.00, 0.15)  # Minimal brood
        
        # Generate realistic inspection notes
        notes = generate_inspection_notes(season, base_photos, bee_confidence)
        
        # Top API label (most common detection)
        top_labels = ["Honeybee", "Insect", "Food", "Pattern", "Animal"]
        weights = [0.4, 0.25, 0.15, 0.1, 0.1]
        top_label = np.random.choice(top_labels, p=weights)
        top_label_confidence = bee_confidence + random.uniform(-0.1, 0.1)
        top_label_confidence = max(0.3, min(0.95, top_label_confidence))  # Clamp to realistic range
        
        inspections.append({
            'inspection_id': i + 1,
            'date': date,
            'season': season,
            'photo_count': base_photos,
            'bee_confidence': bee_confidence,
            'honey_area': honey_area,
            'brood_area': brood_area,
            'notes': notes,
            'top_label': top_label,
            'top_label_confidence': top_label_confidence,
            'month': month,
            'year': date.year
        })
    
    return pd.DataFrame(inspections)

def get_season(month):
    """Convert month number to season"""
    if month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    elif month in [9, 10, 11]:
        return "Fall"
    else:
        return "Winter"

def generate_inspection_notes(season, photo_count, bee_confidence):
    """Generate realistic inspection notes based on conditions"""
    
    base_notes = {
        "Spring": [
            "Checking winter survival", "Queen laying well", "Building up nicely", 
            "Added super", "Lots of activity", "Pollen coming in"
        ],
        "Summer": [
            "Honey flow active", "Full supers", "Regular check", "All looks good",
            "Busy foragers", "Capped honey visible"
        ],
        "Fall": [
            "Harvest prep", "Reducing entrance", "Mite treatment", "Winter prep",
            "Checking stores", "Queen still laying"
        ],
        "Winter": [
            "Quick peek", "Cluster looks good", "Minimal disturbance", 
            "Checking activity", "Still alive!"
        ]
    }
    
    # High photo count suggests interesting events
    if photo_count > 15:
        special_notes = [
            "Queen sighting!", "Swarm cells found", "Unusual behavior", 
            "Great photo opportunities", "Detailed documentation", "Teaching moment"
        ]
        return random.choice(special_notes)
    
    # Low bee confidence might indicate issues
    if bee_confidence < 0.6:
        concern_notes = [
            "Low activity", "Hard to spot bees", "Checking for issues",
            "Quiet hive", "Weather impact?"
        ]
        return random.choice(concern_notes)
    
    return random.choice(base_notes[season])

def create_main_timeline(df):
    """Create the main interactive timeline visualization"""
    
    # Create the timeline scatter plot
    fig = go.Figure()
    
    # Add scatter points for each inspection
    fig.add_trace(go.Scatter(
        x=df['date'],
        y=[1] * len(df),  # All points on same horizontal line
        mode='markers',
        marker=dict(
            size=df['photo_count'],  # Size based on photo count
            color=df['bee_confidence'],  # Color based on bee confidence
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(
                title="Bee Detection<br>Confidence",
            ),
            sizemode='area',
            sizeref=2.*max(df['photo_count'])/(40.**2),  # Scale marker sizes
            sizemin=4,
            line=dict(width=2, color='DarkSlateGrey'),
            opacity=0.8
        ),
        text=df.apply(lambda row: 
            f"<b>{row['date'].strftime('%B %d, %Y')}</b><br>" +
            f"Photos: {row['photo_count']}<br>" +
            f"Bee Confidence: {row['bee_confidence']:.2f}<br>" +
            f"Notes: {row['notes']}<br>" +
            f"Season: {row['season']}", axis=1),
        hovertemplate='%{text}<extra></extra>',
        name='Inspections'
    ))
    
    # Add vertical lines from x-axis to points (lollipop stems)
    for _, row in df.iterrows():
        fig.add_trace(go.Scatter(
            x=[row['date'], row['date']],
            y=[0, 1],
            mode='lines',
            line=dict(color='lightgray', width=2),
            showlegend=False,
            hoverinfo='skip'
        ))
    
    # Customize layout
    fig.update_layout(
        title={
            'text': "🐝 Beehive Inspection Timeline (2020-2024)",
            'x': 0.5,
            'font': {'size': 20}
        },
        xaxis=dict(
            title="Date",
            showgrid=True,
            gridwidth=1,
            gridcolor='lightgray'
        ),
        yaxis=dict(
            title="",
            showticklabels=False,
            range=[-0.2, 1.5],
            showgrid=False
        ),
        plot_bgcolor='white',
        paper_bgcolor='white',
        height=500,
        margin=dict(l=50, r=50, t=80, b=50),
        hovermode='closest'
    )
    
    # Add annotations for seasons
    season_colors = {'Spring': '#90EE90', 'Summer': '#FFD700', 'Fall': '#FFA500', 'Winter': '#87CEEB'}
    for season, color in season_colors.items():
        season_data = df[df['season'] == season]
        if not season_data.empty:
            fig.add_annotation(
                x=season_data['date'].iloc[len(season_data)//2],
                y=1.3,
                text=f"{season}",
                showarrow=False,
                font=dict(color=color, size=12, family="Arial Black"),
                bgcolor=color,
                bordercolor="white",
                borderwidth=1,
                opacity=0.8
            )
    
    return fig

def create_seasonal_patterns(df):
    """Create visualization showing seasonal inspection patterns"""
    
    # Group by month and count inspections
    monthly_counts = df.groupby('month').size().reset_index(name='inspection_count')
    monthly_counts['month_name'] = pd.to_datetime(monthly_counts['month'], format='%m').dt.month_name()
    
    # Create bar chart
    fig = px.bar(
        monthly_counts,
        x='month_name',
        y='inspection_count',
        color='inspection_count',
        color_continuous_scale='Viridis',
        title="📅 Inspection Frequency by Month",
        labels={'month_name': 'Month', 'inspection_count': 'Number of Inspections'}
    )
    
    # Customize layout
    fig.update_layout(
        height=400,
        showlegend=False,
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    
    return fig

def create_photo_activity_correlation(df):
    """Create scatter plot showing photo count vs hive activity indicators"""
    
    fig = px.scatter(
        df,
        x='photo_count',
        y='bee_confidence',
        size='honey_area',
        color='season',
        hover_data=['date', 'notes'],
        title="📸 Photo Count vs Hive Activity",
        labels={
            'photo_count': 'Photos Taken',
            'bee_confidence': 'Bee Detection Confidence',
            'honey_area': 'Honey Area Estimate'
        }
    )
    
    # Add trend line
    fig.add_trace(go.Scatter(
        x=df['photo_count'],
        y=np.poly1d(np.polyfit(df['photo_count'], df['bee_confidence'], 1))(df['photo_count']),
        mode='lines',
        name='Trend',
        line=dict(dash='dash', color='red')
    ))
    
    fig.update_layout(
        height=450,
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    
    return fig

def generate_all_visualizations(output_dir="assets/visualizations/beehive"):
    """Generate all visualizations for the blog post"""
    
    # Create output directory
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Generate sample data
    print("📊 Generating sample inspection data...")
    df = generate_sample_inspection_data()
    
    print(f"Generated {len(df)} inspections from {df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')}")
    
    # Create visualizations
    print("🎨 Creating main timeline...")
    timeline_fig = create_main_timeline(df)
    timeline_fig.write_html(
        f"{output_dir}/inspection-timeline.html",
        include_plotlyjs='cdn',
        config={'displayModeBar': True, 'displaylogo': False}
    )
    
    print("📅 Creating seasonal patterns...")
    patterns_fig = create_seasonal_patterns(df)
    patterns_fig.write_html(
        f"{output_dir}/inspection-patterns.html",
        include_plotlyjs='cdn',
        config={'displayModeBar': True, 'displaylogo': False}
    )
    
    print("📸 Creating photo activity correlation...")
    correlation_fig = create_photo_activity_correlation(df)
    correlation_fig.write_html(
        f"{output_dir}/photos-vs-events.html",
        include_plotlyjs='cdn',
        config={'displayModeBar': True, 'displaylogo': False}
    )
    
    # Save data for reference
    df.to_csv(f"{output_dir}/sample_inspection_data.csv", index=False)
    
    print(f"✅ All visualizations saved to {output_dir}/")
    print("\n📋 Files created:")
    print(f"  - inspection-timeline.html (main timeline)")
    print(f"  - inspection-patterns.html (seasonal patterns)")
    print(f"  - photos-vs-events.html (photo correlation)")
    print(f"  - sample_inspection_data.csv (data for reference)")
    
    print(f"\n🔗 To use in your blog post:")
    print(f'<iframe src="/assets/visualizations/beehive/inspection-timeline.html" width="100%" height="500" frameborder="0"></iframe>')
    
    return df

def create_vision_api_process_diagram():
    """Create a diagram showing the Vision API analysis process"""
    
    # Create a flowchart-style diagram
    fig = go.Figure()
    
    # Define process steps
    steps = [
        {"name": "Photo Upload", "x": 1, "y": 5, "color": "#FF9999"},
        {"name": "Object Detection", "x": 3, "y": 6, "color": "#99CCFF"},
        {"name": "Label Classification", "x": 3, "y": 5, "color": "#99CCFF"},
        {"name": "Color Analysis", "x": 3, "y": 4, "color": "#99CCFF"},
        {"name": "Text Recognition", "x": 3, "y": 3, "color": "#99CCFF"},
        {"name": "Pattern Detection", "x": 3, "y": 2, "color": "#99CCFF"},
        {"name": "Web Entity Matching", "x": 3, "y": 1, "color": "#99CCFF"},
        {"name": "Beekeeping Intelligence", "x": 5, "y": 3.5, "color": "#99FF99"},
        {"name": "Results & Insights", "x": 7, "y": 3.5, "color": "#FFFF99"}
    ]
    
    # Add nodes
    for step in steps:
        fig.add_trace(go.Scatter(
            x=[step["x"]],
            y=[step["y"]],
            mode='markers+text',
            marker=dict(size=80, color=step["color"], line=dict(width=2, color='black')),
            text=step["name"],
            textposition="middle center",
            textfont=dict(size=10, color='black'),
            showlegend=False,
            hoverinfo='text',
            hovertext=step["name"]
        ))
    
    # Add arrows (connections)
    arrows = [
        (1, 5, 3, 6), (1, 5, 3, 5), (1, 5, 3, 4), (1, 5, 3, 3), (1, 5, 3, 2), (1, 5, 3, 1),  # Upload to API endpoints
        (3, 6, 5, 3.5), (3, 5, 5, 3.5), (3, 4, 5, 3.5), (3, 3, 5, 3.5), (3, 2, 5, 3.5), (3, 1, 5, 3.5),  # API to Intelligence
        (5, 3.5, 7, 3.5)  # Intelligence to Results
    ]
    
    for x1, y1, x2, y2 in arrows:
        fig.add_annotation(
            x=x2, y=y2,
            ax=x1, ay=y1,
            xref='x', yref='y',
            axref='x', ayref='y',
            arrowhead=2,
            arrowsize=1,
            arrowwidth=2,
            arrowcolor='gray'
        )
    
    fig.update_layout(
        title="🤖 Computer Vision Analysis Pipeline",
        xaxis=dict(showgrid=False, showticklabels=False, range=[0, 8]),
        yaxis=dict(showgrid=False, showticklabels=False, range=[0, 7]),
        plot_bgcolor='white',
        paper_bgcolor='white',
        height=400,
        margin=dict(l=20, r=20, t=50, b=20)
    )
    
    return fig
    

In [19]:
# if __name__ == "__main__":
# Generate all visualizations
sample_data = generate_all_visualizations()


📊 Generating sample inspection data...
Generated 65 inspections from 2020-03-13 to 2023-11-04
🎨 Creating main timeline...
📅 Creating seasonal patterns...
📸 Creating photo activity correlation...
✅ All visualizations saved to assets/visualizations/beehive/

📋 Files created:
  - inspection-timeline.html (main timeline)
  - inspection-patterns.html (seasonal patterns)
  - photos-vs-events.html (photo correlation)
  - sample_inspection_data.csv (data for reference)

🔗 To use in your blog post:
<iframe src="/assets/visualizations/beehive/inspection-timeline.html" width="100%" height="500" frameborder="0"></iframe>


In [20]:
# Also create the API process diagram
api_fig = create_vision_api_process_diagram()
api_fig.write_html(
    "assets/visualizations/beehive/vision-api-process.html",
    include_plotlyjs='cdn',
    config={'displayModeBar': True, 'displaylogo': False}
)

print("\n🎯 Sample data summary:")
print(f"Total inspections: {len(sample_data)}")
print(f"Date range: {sample_data['date'].min().strftime('%Y-%m-%d')} to {sample_data['date'].max().strftime('%Y-%m-%d')}")
print(f"Average photos per inspection: {sample_data['photo_count'].mean():.1f}")
print(f"Seasons covered: {', '.join(sample_data['season'].unique())}")

print("\n🚀 Ready to embed in your blog post!")



🎯 Sample data summary:
Total inspections: 65
Date range: 2020-03-13 to 2023-11-04
Average photos per inspection: 13.4
Seasons covered: Spring, Summer, Fall, Winter

🚀 Ready to embed in your blog post!
